# 21 · Estadística y probabilidad para Data Science

Machine Learning no reemplaza estadística. Necesitamos cuantificar incertidumbre, entender muestreo, comparar grupos y evitar conclusiones falsas.

## Objetivos
- Repasar variables aleatorias y distribuciones.
- Entender esperanza, varianza, covarianza y correlación.
- Simular LLN y CLT.
- Construir intervalos de confianza paramétricos y bootstrap.
- Usar permutation tests.
- Entender p-values, effect size y multiple testing.
- Introducir Bayes y actualización probabilística.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
SEED=42; rng=np.random.default_rng(SEED)

## 1. Variables aleatorias y momentos
$E[X]$ resume centro esperado; $Var(X)=E[(X-E[X])^2]$ mide dispersión cuadrática. Covarianza mide co-movimiento; correlación normaliza a [-1,1]. Correlación de Pearson detecta asociación lineal; Spearman trabaja con rangos y relaciones monótonas.

Distribuciones frecuentes:
- Bernoulli/Binomial: eventos sí/no y conteos de éxitos.
- Poisson: conteos de eventos bajo supuestos específicos.
- Normal: errores/agregaciones; no todos los datos son gaussianos.
- Exponencial: tiempos entre eventos Poisson.
- Beta: probabilidades/proporciones, útil en Bayes.


In [ ]:
x=np.linspace(-4,4,500); plt.plot(x,stats.norm.pdf(x),label='Normal'); plt.plot(x[x>=0],stats.expon.pdf(x[x>=0]),label='Exponencial'); plt.legend(); plt.show()

## 2. Ley de los Grandes Números
El promedio muestral converge hacia la esperanza cuando crece n bajo condiciones adecuadas. Esto no implica que toda muestra grande sea representativa: sesgo de selección no desaparece aumentando n.


In [ ]:
draws=rng.exponential(scale=2,size=10000); running=np.cumsum(draws)/np.arange(1,len(draws)+1); plt.plot(running); plt.axhline(2,ls='--'); plt.ylim(1.5,2.5); plt.title('LLN: media acumulada'); plt.show()

## 3. Teorema Central del Límite
La distribución de la media de muchas muestras tiende a normalidad bajo condiciones amplias, aunque la población original sea sesgada. Esto sustenta muchos intervalos/tests clásicos. No justifica ignorar dependencia temporal, clusters o colas extremas.


In [ ]:
fig,ax=plt.subplots(1,3,figsize=(13,4))
for a,n in zip(ax,[2,10,50]):
 means=[rng.exponential(2,n).mean() for _ in range(5000)]; a.hist(means,bins=35,density=True); a.set_title(f'medias n={n}')
plt.show()

## 4. Intervalos de confianza
Un IC frecuentista 95% no significa que haya 95% de probabilidad de que el parámetro fijo esté dentro de este intervalo particular. Significa que el procedimiento cubriría el parámetro en 95% de repeticiones bajo los supuestos.


In [ ]:
sample=rng.normal(100,15,120); mean=sample.mean(); se=stats.sem(sample); ci=stats.t.interval(.95,df=len(sample)-1,loc=mean,scale=se); print('mean',mean,'95% CI',ci)

## 5. Bootstrap
Re-muestrea observaciones con reemplazo para aproximar la distribución de un estimador. Sirve para mediana, diferencias, métricas ML y estimadores donde la fórmula analítica es difícil. Si los datos tienen clusters/tiempo, necesitamos block/cluster bootstrap.


In [ ]:
boot=np.array([np.median(rng.choice(sample,size=len(sample),replace=True)) for _ in range(5000)]); print('mediana',np.median(sample),'bootstrap CI',np.quantile(boot,[.025,.975])); plt.hist(boot,bins=40); plt.show()

## 6. Hypothesis testing y p-value
Un p-value es la probabilidad, **asumiendo H0**, de observar un estadístico tan extremo o más que el observado. No es $P(H0|datos)$.

Siempre acompaña significancia con effect size, intervalo y contexto. Un efecto minúsculo puede ser significativo con n enorme; uno útil puede no serlo con poca muestra.


In [ ]:
a=rng.normal(0,.9,300); b=rng.normal(.18,.9,300); print(stats.ttest_ind(a,b,equal_var=False))
pooled=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2)); print('Cohen d',(b.mean()-a.mean())/pooled)

## 7. Permutation test
Bajo H0 de intercambiabilidad, mezclamos labels y recalculamos el efecto. Tiene menos supuestos paramétricos y ayuda a entender la lógica de un test.


In [ ]:
obs=b.mean()-a.mean(); combined=np.r_[a,b]; diffs=[]
for _ in range(5000):
 z=rng.permutation(combined); diffs.append(z[len(a):].mean()-z[:len(a)].mean())
p_perm=np.mean(np.abs(diffs)>=abs(obs)); print('observado',obs,'permutation p',p_perm)

## 8. Correlación y Simpson's paradox
Una relación agregada puede invertirse al controlar por subgrupos. Siempre inspecciona estructura de datos, selection bias y variables de confusión. Pearson=0 tampoco significa independencia en relaciones no lineales.

## 9. Bayes
$$P(H|D)=\frac{P(D|H)P(H)}{P(D)}$$
Bayes combina prior y likelihood. Ejemplo: una prueba con sensibilidad/especificidad altas puede tener bajo valor predictivo positivo si la prevalencia es muy baja.


In [ ]:
prev=.01; sens=.95; spec=.95
ppv=sens*prev/(sens*prev+(1-spec)*(1-prev)); print('PPV con prevalencia 1%:',ppv)
# Beta-Binomial: prior Beta(1,1), observar 8 éxitos de 10 → posterior Beta(9,3)
post=stats.beta(9,3); print('posterior mean',post.mean(),'95% credible interval',post.ppf([.025,.975]))

## 10. Multiple testing
Si pruebas suficientes hipótesis, aparecerán p<0.05 por azar. Bonferroni controla family-wise error; Benjamini-Hochberg controla FDR. Pre-registro de hipótesis y transparencia reducen p-hacking.

## Temas relacionados que seguiremos
Bayesian ML, uncertainty quantification, Monte Carlo, MCMC, causal inference, survival analysis, hierarchical models y conformal prediction.

## Ejercicios
1. Simula coverage de un IC 95% en 5000 experimentos.
2. Muestra CLT para una Bernoulli muy sesgada.
3. Compara bootstrap percentile vs BCa conceptualmente.
4. Simula Simpson's paradox.
5. Calcula PPV para prevalencias 0.1%, 1%, 10%.
6. Implementa Benjamini-Hochberg.
7. Usa bootstrap para IC de ROC-AUC.
8. Investiga Mann-Whitney, chi-square y Fisher exact test.
